# 03 Modeling and Hyperparameter Tuning

This notebook trains multiple regression models, compares split strategies, tunes selected models, and saves the best-performing artifacts.

## Modeling Goals

- train at least four regression models
- compare random split with chronological split
- tune at least two models
- identify the best model using RMSE and R2 Score

In [ ]:
import joblib
import pandas as pd


from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "data").exists() or (candidate / "climate_change_prediction_dataset.csv").exists():
            return candidate
    return current

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed" if (DATA_DIR / "processed").exists() else PROJECT_ROOT
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables" if (OUTPUTS_DIR / "tables").exists() else PROJECT_ROOT
MODELS_DIR = OUTPUTS_DIR / "models"
RAW_DATASET_PATH = RAW_DIR / "GlobalLandTemperaturesByCity.csv"
if not RAW_DATASET_PATH.exists() and (PROJECT_ROOT / "climate_change_prediction_dataset.csv").exists():
    RAW_DATASET_PATH = PROJECT_ROOT / "climate_change_prediction_dataset.csv"

def load_temperature_data(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset not found at {csv_path}")
    return pd.read_csv(csv_path)

TABLES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
from pathlib import Path

import pandas as pd


def format_results_table(results: pd.DataFrame) -> pd.DataFrame:
    ordered = results.copy()
    ordered["MAE"] = ordered["mae"].round(3)
    ordered["MSE"] = ordered["mse"].round(3)
    ordered["RMSE"] = ordered["rmse"].round(3)
    ordered["MAPE"] = ordered["mape"].round(3)
    ordered["R2 Score"] = ordered["r2"].round(3)
    ordered = ordered.rename(
        columns={
            "split_strategy": "Split Strategy",
            "model": "Model",
            "tuned": "Tuned",
        }
    )
    return ordered[["Split Strategy", "Model", "Tuned", "MAE", "MSE", "RMSE", "MAPE", "R2 Score"]]


def save_table(df: pd.DataFrame, path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

from dataclasses import dataclass
import json

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
except ImportError:  # pragma: no cover - optional dependency
    XGBRegressor = None


NUMERIC_FEATURES = [
    "Year",
    "Month",
    "Quarter",
    "YearsSince1900",
    "LatitudeValue",
    "LongitudeValue",
    "AverageTemperatureUncertainty",
    "HistoricalCityMonthMean",
    "Lag1Temperature",
    "Lag12Temperature",
    "Rolling12MeanTemperature",
]
CATEGORICAL_FEATURES = ["Country", "Season", "Hemisphere", "RegionTag"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET_COLUMN = "AverageTemperature"


@dataclass
class ExperimentResult:
    split_strategy: str
    model: str
    tuned: str
    mae: float
    mse: float
    rmse: float
    mape: float
    r2: float


BASELINE_TO_TUNED = {
    "Random Forest": "Random Forest Tuned",
    "Gradient Boosting": "Gradient Boosting Tuned",
}


def build_feature_pipeline() -> ColumnTransformer:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, NUMERIC_FEATURES),
            ("cat", categorical_pipeline, CATEGORICAL_FEATURES),
        ]
    )


def prepare_model_frame(df: pd.DataFrame) -> pd.DataFrame:
    return df.dropna(subset=NUMERIC_FEATURES + [TARGET_COLUMN]).copy()


def split_random(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
    return train_df.copy(), test_df.copy()


def split_chronological(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = df.sort_values("dt").reset_index(drop=True)
    split_index = int(len(ordered) * 0.8)
    train_df = ordered.iloc[:split_index].copy()
    test_df = ordered.iloc[split_index:].copy()
    return train_df, test_df


def _model_catalog() -> dict[str, object]:
    models: dict[str, object] = {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=14, min_samples_leaf=8, random_state=42),
        "Random Forest": RandomForestRegressor(
            n_estimators=120,
            max_depth=16,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1,
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ),
    }
    if XGBRegressor is not None:
        models["XGBoost"] = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=120,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )
    return models


def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    mse = mean_squared_error(y_true, y_pred)
    denominator = np.maximum(np.abs(np.asarray(y_true, dtype="float64")), 1e-3)
    mape = np.mean(np.abs((np.asarray(y_pred, dtype="float64") - np.asarray(y_true, dtype="float64")) / denominator)) * 100
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "mse": float(mse),
        "rmse": float(np.sqrt(mse)),
        "mape": float(mape),
        "r2": float(r2_score(y_true, y_pred)),
    }


def fit_model(train_df: pd.DataFrame, estimator: object) -> Pipeline:
    pipeline = Pipeline(
        steps=[
            ("preprocessor", build_feature_pipeline()),
            ("model", estimator),
        ]
    )
    pipeline.fit(train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN])
    return pipeline


def run_model_suite(df: pd.DataFrame, split_strategy: str) -> tuple[pd.DataFrame, dict[str, Pipeline]]:
    prepared = prepare_model_frame(df)
    if split_strategy == "chronological":
        train_df, test_df = split_chronological(prepared)
    else:
        train_df, test_df = split_random(prepared)

    results: list[ExperimentResult] = []
    fitted_models: dict[str, Pipeline] = {}
    for model_name, estimator in _model_catalog().items():
        pipeline = fit_model(train_df, estimator)
        predictions = pipeline.predict(test_df[FEATURE_COLUMNS])
        metrics = evaluate_predictions(test_df[TARGET_COLUMN], predictions)
        fitted_models[model_name] = pipeline
        results.append(
            ExperimentResult(
                split_strategy=split_strategy,
                model=model_name,
                tuned="No",
                **metrics,
            )
        )

    results_df = pd.DataFrame([result.__dict__ for result in results]).sort_values(
        ["split_strategy", "rmse", "r2"],
        ascending=[True, True, False],
    )
    return results_df, fitted_models


def tune_models(
    train_df: pd.DataFrame,
    sample_size: int = 18_000,
) -> tuple[dict[str, Pipeline], pd.DataFrame]:
    tune_source = train_df.copy()
    if len(tune_source) > sample_size:
        tune_source = tune_source.sample(n=sample_size, random_state=42)

    search_spaces: dict[str, tuple[object, dict[str, list[object]]]] = {
        "Random Forest Tuned": (
            RandomForestRegressor(random_state=42, n_jobs=-1),
            {
                "model__n_estimators": [150, 220, 300],
                "model__max_depth": [12, 16, None],
                "model__min_samples_leaf": [2, 4, 8],
                "model__max_features": ["sqrt", 0.8, None],
            },
        ),
        "Gradient Boosting Tuned": (
            GradientBoostingRegressor(random_state=42),
            {
                "model__n_estimators": [120, 180, 240],
                "model__learning_rate": [0.03, 0.05, 0.08],
                "model__max_depth": [2, 3, 4],
                "model__subsample": [0.8, 1.0],
            },
        ),
    }

    tuned_models: dict[str, Pipeline] = {}
    tuning_rows: list[dict[str, object]] = []
    for model_name, (estimator, params) in search_spaces.items():
        search = RandomizedSearchCV(
            estimator=Pipeline(
                steps=[
                    ("preprocessor", build_feature_pipeline()),
                    ("model", estimator),
                ]
            ),
            param_distributions=params,
            n_iter=4,
            scoring="neg_root_mean_squared_error",
            cv=3,
            random_state=42,
            n_jobs=-1,
            verbose=0,
        )
        search.fit(tune_source[FEATURE_COLUMNS], tune_source[TARGET_COLUMN])
        tuned_models[model_name] = search.best_estimator_
        tuning_rows.append(
            {
                "Model": model_name,
                "Search Method": "RandomizedSearchCV",
                "CV Folds": 3,
                "Iterations": 4,
                "Best CV RMSE": round(float(-search.best_score_), 4),
                "Best Parameters": json.dumps(search.best_params_, sort_keys=True),
            }
        )

    tuning_details = pd.DataFrame(tuning_rows).sort_values("Best CV RMSE").reset_index(drop=True)
    return tuned_models, tuning_details


def evaluate_tuned_models(
    df: pd.DataFrame,
    split_strategy: str,
) -> tuple[pd.DataFrame, dict[str, Pipeline], pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    prepared = prepare_model_frame(df)
    if split_strategy == "chronological":
        train_df, test_df = split_chronological(prepared)
    else:
        train_df, test_df = split_random(prepared)

    tuned_models, tuning_details = tune_models(train_df)
    results: list[ExperimentResult] = []
    for model_name, pipeline in tuned_models.items():
        predictions = pipeline.predict(test_df[FEATURE_COLUMNS])
        metrics = evaluate_predictions(test_df[TARGET_COLUMN], predictions)
        results.append(
            ExperimentResult(
                split_strategy=split_strategy,
                model=model_name,
                tuned="Yes",
                **metrics,
            )
        )

    results_df = pd.DataFrame([result.__dict__ for result in results]).sort_values("rmse")
    return results_df, tuned_models, train_df, test_df, tuning_details


def build_cross_validation_summary(
    df: pd.DataFrame,
    sample_size: int = 18_000,
    folds: int = 5,
) -> pd.DataFrame:
    prepared = prepare_model_frame(df)
    cv_source = prepared.copy()
    if len(cv_source) > sample_size:
        cv_source = cv_source.sample(n=sample_size, random_state=42)

    cv = KFold(n_splits=folds, shuffle=True, random_state=42)
    rows: list[dict[str, object]] = []
    scoring = {
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    }

    for model_name, estimator in _model_catalog().items():
        pipeline = Pipeline(
            steps=[
                ("preprocessor", build_feature_pipeline()),
                ("model", estimator),
            ]
        )
        scores = cross_validate(
            pipeline,
            cv_source[FEATURE_COLUMNS],
            cv_source[TARGET_COLUMN],
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
        )
        rows.append(
            {
                "Model": model_name,
                "CV Folds": folds,
                "Rows Used": len(cv_source),
                "CV Mean MAE": round(float(-scores["test_mae"].mean()), 4),
                "CV Std MAE": round(float(scores["test_mae"].std()), 4),
                "CV Mean RMSE": round(float(-scores["test_rmse"].mean()), 4),
                "CV Std RMSE": round(float(scores["test_rmse"].std()), 4),
                "CV Mean R2": round(float(scores["test_r2"].mean()), 4),
                "CV Std R2": round(float(scores["test_r2"].std()), 4),
            }
        )

    return pd.DataFrame(rows).sort_values("CV Mean RMSE").reset_index(drop=True)


def build_tuning_comparison(
    baseline_results: pd.DataFrame,
    tuned_results: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    baseline_lookup = baseline_results.set_index("model")
    tuned_lookup = tuned_results.set_index("model")

    for baseline_model, tuned_model in BASELINE_TO_TUNED.items():
        if baseline_model not in baseline_lookup.index or tuned_model not in tuned_lookup.index:
            continue

        baseline_rmse = float(baseline_lookup.loc[baseline_model, "rmse"])
        tuned_rmse = float(tuned_lookup.loc[tuned_model, "rmse"])
        baseline_r2 = float(baseline_lookup.loc[baseline_model, "r2"])
        tuned_r2 = float(tuned_lookup.loc[tuned_model, "r2"])

        rows.append(
            {
                "Baseline Model": baseline_model,
                "Tuned Model": tuned_model,
                "Baseline RMSE": round(baseline_rmse, 4),
                "Tuned RMSE": round(tuned_rmse, 4),
                "RMSE Change": round(tuned_rmse - baseline_rmse, 4),
                "Baseline R2": round(baseline_r2, 4),
                "Tuned R2": round(tuned_r2, 4),
                "R2 Change": round(tuned_r2 - baseline_r2, 4),
            }
        )

    return pd.DataFrame(rows)


def extract_feature_importance(pipeline: Pipeline, top_n: int = 15) -> pd.DataFrame:
    preprocessor = pipeline.named_steps["preprocessor"]
    model = pipeline.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()

    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
    elif hasattr(model, "coef_"):
        values = np.abs(np.ravel(model.coef_))
    else:
        return pd.DataFrame(columns=["Feature", "Importance"])

    importance_df = pd.DataFrame(
        {
            "Feature": feature_names,
            "Importance": np.asarray(values, dtype="float64"),
        }
    )
    return importance_df.sort_values("Importance", ascending=False).head(top_n).reset_index(drop=True)


def prediction_frame(
    pipeline: Pipeline,
    test_df: pd.DataFrame,
    split_strategy: str,
    model_name: str,
) -> pd.DataFrame:
    predictions = pipeline.predict(test_df[FEATURE_COLUMNS])
    output = test_df[["dt", "City", "Country", TARGET_COLUMN]].copy()
    output["PredictedTemperature"] = predictions
    output["Residual"] = output[TARGET_COLUMN] - output["PredictedTemperature"]
    output["SplitStrategy"] = split_strategy
    output["Model"] = model_name
    return output.reset_index(drop=True)


def build_error_by_decade(predictions_df: pd.DataFrame) -> pd.DataFrame:
    analysis = predictions_df.copy()
    analysis["Decade"] = ((analysis["dt"].dt.year // 10) * 10).astype(int)
    analysis["AbsoluteError"] = analysis["Residual"].abs()
    analysis["SquaredError"] = analysis["Residual"] ** 2

    grouped = (
        analysis.groupby("Decade", as_index=False)
        .agg(
            Rows=("Residual", "size"),
            MeanAbsoluteError=("AbsoluteError", "mean"),
            RootMeanSquaredError=("SquaredError", lambda values: float(np.sqrt(np.mean(values)))),
            MeanResidual=("Residual", "mean"),
        )
        .sort_values("Decade")
    )
    grouped["MeanAbsoluteError"] = grouped["MeanAbsoluteError"].round(4)
    grouped["RootMeanSquaredError"] = grouped["RootMeanSquaredError"].round(4)
    grouped["MeanResidual"] = grouped["MeanResidual"].round(4)
    return grouped.reset_index(drop=True)


## Load Modeling Dataset

This file is the practical training sample prepared during preprocessing.

In [2]:
data = pd.read_csv(PROCESSED_DIR / 'modeling_dataset.csv', parse_dates=['dt'])
print('Modeling dataset shape:', data.shape)
data.head()

Modeling dataset shape: (75000, 25)


,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude,LatitudeValue,LongitudeValue,Year,...,YearsSince1900,RegionTag,Lag1Temperature,Lag12Temperature,Rolling12MeanTemperature,HistoricalCityMonthMean,TemperatureAnomaly,CityYearMeanTemperature,DecadeMeanTemperature,ClimateRiskIndex
0,1900-07-01,23.470,0.801,Cartagena,Spain,37.78N,0.00W,37.78,-0.00,1900,...,0,Global,29.283,12.283,21.741500,29.283,-5.813001,22.572792,17.346918,-3.813534
1,1900-08-01,26.240,0.364,Colombo,Sri Lanka,7.23N,80.27E,7.23,80.27,1900,...,0,South Asia,15.205,26.290,21.537250,15.205,11.035000,21.940708,17.346918,5.031665
2,1900-09-01,25.975,0.341,Colombo,Sri Lanka,7.23N,80.27E,7.23,80.27,1900,...,0,South Asia,16.276,27.888,21.148000,16.276,9.699001,21.940708,17.346918,4.117771
3,1900-10-01,11.560,0.479,Springfield,United States,42.59N,72.00W,42.59,-72.00,1900,...,0,Global,16.891,18.907,21.491667,16.729,-5.169000,11.294556,17.346918,-4.600680
4,1901-01-01,19.306,0.370,Taunggyi,Burma,20.09N,97.25E,20.09,97.25,1901,...,1,Global,19.812,19.009,23.673334,19.009,0.296999,23.684166,17.346918,-1.610227


## Baseline Models on Random Split

Random split is useful as a benchmark, but it may be optimistic for time-related climate data.

In [3]:
random_results, random_models = run_model_suite(data, split_strategy='random')
random_results

,split_strategy,model,tuned,mae,mse,rmse,r2
2,random,Random Forest,No,0.875363,1.720494,1.311676,0.982795
4,random,XGBoost,No,0.890146,1.743060,1.320250,0.982569
1,random,Decision Tree,No,0.958450,2.119022,1.455686,0.978809
3,random,Gradient Boosting,No,0.946740,2.153778,1.467575,0.978462
0,random,Linear Regression,No,0.995313,2.426533,1.557733,0.975734


Interpretation: Random split usually gives slightly better metrics because past and future patterns can mix more easily.

## Baseline Models on Chronological Split

Chronological split is more realistic because the model is trained on earlier data and tested on later data.

In [4]:
chrono_results, chrono_models = run_model_suite(data, split_strategy='chronological')
chrono_results

,split_strategy,model,tuned,mae,mse,rmse,r2
2,chronological,Random Forest,No,0.927169,1.882004,1.371861,0.980485
4,chronological,XGBoost,No,0.932164,1.894259,1.376321,0.980358
3,chronological,Gradient Boosting,No,0.964709,2.126050,1.458098,0.977954
0,chronological,Linear Regression,No,1.009243,2.390529,1.546133,0.975211
1,chronological,Decision Tree,No,1.040459,2.425386,1.557365,0.974850


Interpretation: These results are more suitable for the final report because they better reflect future-style prediction.

## Hyperparameter Tuning

The notebook tunes two stronger models: Random Forest and Gradient Boosting.

In [5]:
tuned_results, tuned_models, train_df, test_df, tuning_details = evaluate_tuned_models(data, split_strategy='chronological')
tuned_results

,split_strategy,model,tuned,mae,mse,rmse,r2
1,chronological,Gradient Boosting Tuned,Yes,0.959657,2.126478,1.458245,0.977950
0,chronological,Random Forest Tuned,Yes,0.981599,2.230528,1.493495,0.976871


Interpretation: Tuning improves the rigor of the workflow, even if the untuned Random Forest remains the strongest final baseline.

## Combined Comparison Table

This is the main result table used in the report.

In [6]:
combined_results = pd.concat([random_results, chrono_results, tuned_results], ignore_index=True)
comparison_table = format_results_table(combined_results)
comparison_table.to_csv(TABLES_DIR / 'model_comparison_results.csv', index=False)
comparison_table

,Split Strategy,Model,Tuned,MAE,MSE,RMSE,R2 Score
0,random,Random Forest,No,0.875,1.720,1.312,0.983
1,random,XGBoost,No,0.890,1.743,1.320,0.983
2,random,Decision Tree,No,0.958,2.119,1.456,0.979
3,random,Gradient Boosting,No,0.947,2.154,1.468,0.978
4,random,Linear Regression,No,0.995,2.427,1.558,0.976
5,chronological,Random Forest,No,0.927,1.882,1.372,0.980
6,chronological,XGBoost,No,0.932,1.894,1.376,0.980
7,chronological,Gradient Boosting,No,0.965,2.126,1.458,0.978
8,chronological,Linear Regression,No,1.009,2.391,1.546,0.975
9,chronological,Decision Tree,No,1.040,2.425,1.557,0.975


## Best Hyperparameters and Before/After Tuning

These tables make the tuning stage clearer for the report.

In [7]:
tuning_details.to_csv(TABLES_DIR / 'best_hyperparameters.csv', index=False)
tuning_details

,Model,Search Method,CV Folds,Iterations,Best CV RMSE,Best Parameters
0,Gradient Boosting Tuned,RandomizedSearchCV,3,4,1.4225,"{""model__learning_rate"": 0.08, ""model__max_dep..."
1,Random Forest Tuned,RandomizedSearchCV,3,4,1.4889,"{""model__max_depth"": 12, ""model__max_features""..."


In [8]:
chrono_baseline_subset = chrono_results[chrono_results['model'].isin(['Random Forest', 'Gradient Boosting'])].copy()
tuning_comparison = build_tuning_comparison(chrono_baseline_subset, tuned_results)
tuning_comparison.to_csv(TABLES_DIR / 'tuning_before_after_comparison.csv', index=False)
tuning_comparison

,Baseline Model,Tuned Model,Baseline RMSE,Tuned RMSE,RMSE Change,Baseline R2,Tuned R2,R2 Change
0,Random Forest,Random Forest Tuned,1.3719,1.4935,0.1216,0.9805,0.9769,-0.0036
1,Gradient Boosting,Gradient Boosting Tuned,1.4581,1.4582,0.0001,0.9780,0.9779,-0.0000


## Best Model Selection

The best chronological baseline and best tuned model are saved for later evaluation.

In [9]:
best_baseline_name = chrono_results.sort_values('rmse').iloc[0]['model']
best_tuned_name = tuned_results.sort_values('rmse').iloc[0]['model']

joblib.dump(chrono_models[best_baseline_name], MODELS_DIR / 'best_baseline_model.joblib')
joblib.dump(tuned_models[best_tuned_name], MODELS_DIR / 'best_tuned_model.joblib')
train_df.to_csv(PROCESSED_DIR / 'chronological_train_split.csv', index=False)
test_df.to_csv(PROCESSED_DIR / 'chronological_test_split.csv', index=False)

print('Best baseline model:', best_baseline_name)
print('Best tuned model:', best_tuned_name)

Best baseline model: Random Forest
Best tuned model: Gradient Boosting Tuned


Final note: For this project, Random Forest under chronological evaluation is the strongest main result.